# Mean-Variance and Black-Litterman Portfolio Optimization

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import statsmodels.api as sm
from statsmodels.robust.robust_linear_model import RLM
from statsmodels.robust import norms as rnorms

np.random.seed(42)
sns.set_style("whitegrid")

C_INDIGO  = "#4F46E5"
C_EMERALD = "#10B981"
C_AMBER   = "#F59E0B"
C_ROSE    = "#F43F5E"
C_SLATE   = "#94A3B8"
PALETTE10 = ["#4F46E5","#10B981","#F59E0B","#F43F5E","#94A3B8",
              "#8B5CF6","#EC4899","#14B8A6","#F97316","#84CC16"]

TICKERS      = ['TSLA','WMT','BAC','GS','LLY','MRK','GOOG','META','AAPL','XOM']
TRAIN_START  = '2023-09-01'
TRAIN_END    = '2025-09-30'
TEST_START   = '2025-10-01'
TEST_END     = '2025-12-31'
RF_ANNUAL    = 0.05
RF_DAILY     = RF_ANNUAL / 252
N_ASSETS     = len(TICKERS)
TRADING_DAYS = 252

print("Libraries loaded.")

### Data Collection

In [ ]:
try:
    import yfinance as yf
    _all = yf.download(TICKERS + ['SPY'], start=TRAIN_START, end=TRAIN_END,
                       auto_adjust=True, progress=False)['Close']
    if isinstance(_all.columns, pd.MultiIndex):
        _all.columns = _all.columns.get_level_values(0)
    _all.index = pd.DatetimeIndex(_all.index).tz_localize(None)
    raw_train = _all[TICKERS].dropna()
    spy_train = _all['SPY'].reindex(raw_train.index).ffill()

    _te = yf.download(TICKERS, start=TEST_START, end=TEST_END,
                      auto_adjust=True, progress=False)['Close']
    if isinstance(_te.columns, pd.MultiIndex):
        _te.columns = _te.columns.get_level_values(0)
    _te.index = pd.DatetimeIndex(_te.index).tz_localize(None)
    raw_test = _te[TICKERS].dropna()
    if len(raw_train) < 100:
        raise ValueError("Insufficient data")
    DATA_SRC = "yfinance (live)"

except Exception as e:
    print(f"yfinance unavailable ({type(e).__name__}) - generating synthetic data")
    rng = np.random.default_rng(42)
    n_tr = 522
    dates_tr = pd.bdate_range(TRAIN_START, periods=n_tr)
    mu_arr  = np.array([0.10,0.09,0.09,0.12,0.25,0.09,0.13,0.28,0.15,0.07])
    vol_arr = np.array([0.55,0.20,0.28,0.30,0.35,0.20,0.30,0.45,0.28,0.22])
    C_ = 0.30*np.ones((10,10)) + 0.70*np.eye(10)
    L_ = np.linalg.cholesky(C_)
    Z_ = rng.standard_normal((n_tr,10)) @ L_.T
    lr = (mu_arr - 0.5*vol_arr**2)/252 + vol_arr/np.sqrt(252)*Z_
    raw_train = pd.DataFrame(100*np.exp(np.cumsum(lr,0)), index=dates_tr, columns=TICKERS)
    spy_w = np.array([0.018,0.012,0.009,0.005,0.021,0.010,0.040,0.027,0.072,0.015])
    spy_log = (lr * spy_w).sum(axis=1)
    spy_train = pd.Series(420*np.exp(np.cumsum(spy_log)), index=dates_tr, name='SPY')
    n_te = 63
    dates_te = pd.bdate_range(TEST_START, periods=n_te)
    Z_te = rng.standard_normal((n_te,10)) @ L_.T
    lr_te = (mu_arr - 0.5*vol_arr**2)/252 + vol_arr/np.sqrt(252)*Z_te
    raw_test = pd.DataFrame(raw_train.iloc[-1].values * np.exp(np.cumsum(lr_te,0)),
                            index=dates_te, columns=TICKERS)
    DATA_SRC = "Synthetic (yfinance unavailable)"

ret_train = raw_train.pct_change().dropna()
ret_test  = raw_test.pct_change().dropna()
spy_ret   = spy_train.pct_change().dropna().reindex(ret_train.index).fillna(0)

T_TR    = len(ret_train)
mu_ann  = ret_train.mean()  * TRADING_DAYS
cov_ann = ret_train.cov()   * TRADING_DAYS
mu_vec  = mu_ann.values
cov_mat = cov_ann.values

print(f"Source   : {DATA_SRC}")
print(f"Training : {T_TR} obs  ({raw_train.index[0].date()} -> {raw_train.index[-1].date()})")
print(f"Test     : {len(ret_test)} obs  ({raw_test.index[0].date()} -> {raw_test.index[-1].date()})")
print(f"\nAnnualised Mean Returns:")
for t, m in zip(TICKERS, mu_ann):
    print(f"  {t:<5}: {m:+.4f}  ({m:.2%})")

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(ret_train.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.3, linecolor='white')
ax.set_title('Asset Return Correlation Matrix (Training Period)')
plt.tight_layout(); plt.show()

## Part 1 - Standard Mean-Variance Portfolio Optimization

In [ ]:
# ── Core optimization functions ───────────────────────────────────────────────
def portfolio_stats(w, mu, cov, rf=RF_ANNUAL):
    ret    = float(w @ mu)
    vol    = float(np.sqrt(max(w @ cov @ w, 1e-12)))
    sharpe = (ret - rf) / vol
    return ret, vol, sharpe

def max_sharpe_opt(mu, cov, rf=RF_ANNUAL, ub=None, n_starts=30):
    """
    Find the tangency (max Sharpe) portfolio by maximising (mu - rf) / sigma.
    Uses SLSQP with multiple random starts to find the global optimum.
    Constraints: weights sum to 1, weights >= 0, weights <= ub (default 1).
    """
    n  = len(mu)
    ub = [1.0]*n if ub is None else list(ub)
    bounds      = [(0.0, u) for u in ub]
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]

    def neg_sharpe(w):
        r = float(w @ mu)
        v = float(np.sqrt(max(w @ cov @ w, 1e-12)))
        return -(r - rf) / v

    best, best_val = None, np.inf
    rng_s = np.random.default_rng(0)
    for _ in range(n_starts):
        w0 = rng_s.dirichlet(np.ones(n))
        w0 = np.clip(w0, 0, ub); w0 /= w0.sum()
        res = minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds,
                       constraints=constraints, options={'ftol':1e-9,'maxiter':10000})
        if res.success and res.fun < best_val:
            best_val, best = res.fun, res
    if best is None:
        best = minimize(neg_sharpe, np.ones(n)/n, method='SLSQP', bounds=bounds,
                        constraints=constraints, options={'ftol':1e-9,'maxiter':10000})
    return best

def efficient_frontier(mu, cov, n_pts=120, ub=None):
    """Compute the efficient frontier by minimising variance for each target return."""
    n  = len(mu)
    ub = [1.0]*n if ub is None else list(ub)
    bounds = [(0.0, u) for u in ub]

    mu_lo   = mu.min() * (0.85 if mu.min() > 0 else 1.15)
    targets = np.linspace(mu_lo, mu.max()*0.98, n_pts)
    vols, rets = [], []
    for tgt in targets:
        cons = [{'type':'eq','fun': lambda w: w.sum()-1},
                {'type':'eq','fun': lambda w, t=tgt: w@mu - t}]
        res = minimize(lambda w: w@cov@w, np.ones(n)/n,
                       method='SLSQP', bounds=bounds, constraints=cons,
                       options={'ftol':1e-9,'maxiter':5000})
        if res.success and res.fun >= 0:
            vols.append(np.sqrt(res.fun));  rets.append(tgt)
    return np.array(vols), np.array(rets)

print("Optimization helper functions defined.")

### Step 1 - Optimal Portfolio (No Short-Selling, Weights Sum to 1)

In [ ]:
# ── Step 1: Tangency portfolio, no short-selling ──────────────────────────────
res_s1 = max_sharpe_opt(mu_vec, cov_mat, rf=RF_ANNUAL)
w_s1   = np.clip(res_s1.x, 0, 1); w_s1 /= w_s1.sum()
ret_s1, vol_s1, sharpe_s1 = portfolio_stats(w_s1, mu_vec, cov_mat)

print("Step 1 - Optimal Tangency Portfolio (No Short-Selling)")
print(f"{'Ticker':<8} {'Weight':>12}")
print("-" * 24)
for t, w in zip(TICKERS, w_s1):
    print(f"{t:<8} {w:>12.4f}   ({w:.2%})")
print(f"\nAnnualised Return  : {ret_s1:.4f}  ({ret_s1:.2%})")
print(f"Annualised Vol     : {vol_s1:.4f}  ({vol_s1:.2%})")
print(f"Sharpe Ratio       : {sharpe_s1:.4f}")
print(f"Non-zero assets    : {(w_s1 > 1e-3).sum()} / {N_ASSETS}")

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(TICKERS, w_s1*100, color=PALETTE10, alpha=0.88, edgecolor='white')
for bar, v in zip(bars, w_s1):
    if v > 0.005:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(10, color=C_SLATE, lw=1.2, ls='--', label='Equal weight (10%)')
ax.set_title('Step 1 - Tangency Portfolio Weights (No Short-Selling, No Cap)')
ax.set_ylabel('Weight (%)'); ax.legend()
plt.tight_layout(); plt.show()

### Step 2 - Optimal Portfolio with 18% Maximum Weight Constraint

In [ ]:
# ── Step 2: Tangency portfolio with max 18% per asset ────────────────────────
MAX_W2 = 0.18
res_s2 = max_sharpe_opt(mu_vec, cov_mat, rf=RF_ANNUAL, ub=[MAX_W2]*N_ASSETS)
w_s2   = np.clip(res_s2.x, 0, MAX_W2); w_s2 /= w_s2.sum()
ret_s2, vol_s2, sharpe_s2 = portfolio_stats(w_s2, mu_vec, cov_mat)

print("Step 2 - Optimal Portfolio (Max 18% Per Asset)")
print(f"{'Ticker':<8} {'Step2 Weight':>14} {'Step1 Weight':>14} {'Change':>10}")
print("-" * 52)
for t, w2, w1 in zip(TICKERS, w_s2, w_s1):
    flag = ' <<CAP' if abs(w2 - MAX_W2) < 1e-3 else ''
    print(f"{t:<8} {w2:>14.4f} {w1:>14.4f} {(w2-w1):>+10.4f}{flag}")
print(f"\nAnnualised Return  : {ret_s2:.4f}  ({ret_s2:.2%})")
print(f"Annualised Vol     : {vol_s2:.4f}  ({vol_s2:.2%})")
print(f"Sharpe Ratio       : {sharpe_s2:.4f}")
print(f"Non-zero assets    : {(w_s2>1e-3).sum()} / {N_ASSETS}")
print(f"Assets at cap      : {(abs(w_s2-MAX_W2)<1e-3).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors1 = [C_INDIGO if w>1e-3 else C_SLATE for w in w_s1]
colors2 = [C_EMERALD if w>1e-3 else C_SLATE for w in w_s2]
axes[0].bar(TICKERS, w_s1*100, color=colors1, alpha=0.88, edgecolor='white')
axes[0].axhline(18, color=C_ROSE, lw=1.5, ls='--', label='18% cap')
axes[0].set_title('Step 1 - Weights (No Cap)'); axes[0].set_ylabel('Weight (%)')
axes[0].legend()
axes[1].bar(TICKERS, w_s2*100, color=colors2, alpha=0.88, edgecolor='white')
axes[1].axhline(18, color=C_ROSE, lw=1.5, ls='--', label='18% cap')
axes[1].set_title('Step 2 - Weights (Max 18% Cap)'); axes[1].set_ylabel('Weight (%)')
axes[1].legend()
plt.suptitle('Step 1 vs Step 2 - Portfolio Weight Comparison', fontsize=12)
plt.tight_layout(); plt.show()

### Step 3 - Efficient Frontier Comparison

In [ ]:
# ── Step 3: Compute both efficient frontiers ──────────────────────────────────
print("Computing unconstrained efficient frontier (Step 1)...")
ef_vol_s1, ef_ret_s1 = efficient_frontier(mu_vec, cov_mat, n_pts=120)

print("Computing constrained efficient frontier (Step 2, max 18%)...")
ef_vol_s2, ef_ret_s2 = efficient_frontier(mu_vec, cov_mat, n_pts=120,
                                           ub=[0.18]*N_ASSETS)
print("Done.")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(ef_vol_s1*100, ef_ret_s1*100, color=C_INDIGO, lw=2.5,
        label='Step 1 - Unconstrained (No Cap)')
ax.plot(ef_vol_s2*100, ef_ret_s2*100, color=C_EMERALD, lw=2.5, ls='--',
        label='Step 2 - Constrained (Max 18% Cap)')

ax.scatter(vol_s1*100, ret_s1*100, s=200, marker='*', color=C_INDIGO, zorder=6,
           label=f'Step 1 Tangency (Sharpe={sharpe_s1:.3f})')
ax.scatter(vol_s2*100, ret_s2*100, s=200, marker='*', color=C_EMERALD, zorder=6,
           label=f'Step 2 Tangency (Sharpe={sharpe_s2:.3f})')

# Individual assets
for i, t in enumerate(TICKERS):
    ax.scatter(np.sqrt(cov_mat[i,i])*100, mu_vec[i]*100,
               color=PALETTE10[i], s=70, alpha=0.7, zorder=5)
    ax.annotate(t, (np.sqrt(cov_mat[i,i])*100+0.2, mu_vec[i]*100),
                fontsize=8.5, color=PALETTE10[i])

# Capital market lines
if len(ef_vol_s1) > 0:
    vol_range = np.linspace(0, ef_vol_s1.max()*100*1.15, 100)
    ax.plot(vol_range, RF_ANNUAL*100 + sharpe_s1*vol_range,
            color=C_INDIGO, lw=1.0, ls=':', alpha=0.5, label='CML (Step 1)')
    ax.plot(vol_range, RF_ANNUAL*100 + sharpe_s2*vol_range,
            color=C_EMERALD, lw=1.0, ls=':', alpha=0.5, label='CML (Step 2)')

ax.axhline(RF_ANNUAL*100, color=C_SLATE, lw=1.0, ls='-.', alpha=0.6,
           label=f'Risk-free ({RF_ANNUAL:.0%})')
ax.set_title('Step 3 - Efficient Frontier Comparison: Unconstrained vs Max 18% Cap',
             fontsize=13)
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

print("""
Step 3 - Discussion:
The main difference is that the constrained efficient frontier (Step 2, max 18%)
lies strictly INSIDE the unconstrained frontier (Step 1) at every risk level:

1. Lower achievable Sharpe ratio: the tangency portfolio Sharpe is lower under
   the cap because the constraint prevents concentration in the highest-Sharpe assets.

2. Narrower return range: the maximum achievable return is capped (cannot put all
   weight in the best-performing asset), and the minimum-variance point shifts
   rightward (diversification is forced, but not necessarily efficiency-improving).

3. The constrained frontier is compressed: both the upper and lower ends of the
   efficient frontier are truncated relative to the unconstrained case.

4. Economically, the 18% cap is a binding constraint that reduces portfolio
   efficiency - it imposes a real cost on the investor measured by the reduction
   in Sharpe ratio from the unconstrained to the constrained tangency portfolio.
""")

### Step 4 - Out-of-Sample Performance (Oct - Dec 2025)

In [ ]:
# ── Out-of-sample performance metrics ────────────────────────────────────────
def oos_performance(weights, ret_df, tickers, rf_daily=RF_DAILY):
    pr      = ret_df[tickers].dropna() @ weights
    cum     = (1 + pr).cumprod()
    tot_ret = cum.iloc[-1] - 1
    ann_ret = pr.mean() * TRADING_DAYS
    ann_vol = pr.std()  * np.sqrt(TRADING_DAYS)
    rf_ann  = rf_daily  * TRADING_DAYS   # annualise the passed daily rf rate
    sharpe  = (ann_ret - rf_ann) / ann_vol if ann_vol > 1e-10 else np.nan
    dd      = (cum / cum.cummax()) - 1
    max_dd  = dd.min()
    return {'port_ret': pr, 'cum': cum, 'drawdown': dd,
            'total_return': tot_ret, 'ann_return': ann_ret,
            'ann_vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd}

oos1 = oos_performance(w_s1, ret_test, TICKERS)
oos2 = oos_performance(w_s2, ret_test, TICKERS)

print("=== OUT-OF-SAMPLE PERFORMANCE  (Oct 1 - Dec 31, 2025) ===")
print(f"\n{'Metric':<28} {'Step1 (No Cap)':>18} {'Step2 (Max 18%)':>18}")
print("-" * 68)
rows = [
    ('Total Return',  'total_return'),
    ('Ann. Return',   'ann_return'),
    ('Ann. Vol',      'ann_vol'),
    ('Sharpe Ratio',  'sharpe'),
    ('Max Drawdown',  'max_dd'),
]
for name, key in rows:
    v1, v2 = oos1[key], oos2[key]
    print(f"{name:<28} {v1:>+18.4f} {v2:>+18.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(oos1['cum'].values, color=C_INDIGO, lw=2.0,
             label=f'Step 1 (No Cap)  Total={oos1["total_return"]:+.2%}')
axes[0].plot(oos2['cum'].values, color=C_EMERALD, lw=2.0, ls='--',
             label=f'Step 2 (Max 18%) Total={oos2["total_return"]:+.2%}')
axes[0].axhline(1.0, color=C_SLATE, lw=0.8, ls=':')
axes[0].set_title('Step 4 - Out-of-Sample Cumulative Return (Oct-Dec 2025)')
axes[0].set_ylabel('Portfolio Value (Base=1)'); axes[0].legend()

axes[1].fill_between(range(len(oos1['drawdown'])), 0, oos1['drawdown'].values,
                      alpha=0.45, color=C_INDIGO, label='Step 1 Drawdown')
axes[1].fill_between(range(len(oos2['drawdown'])), 0, oos2['drawdown'].values,
                      alpha=0.35, color=C_EMERALD, label='Step 2 Drawdown')
axes[1].set_title('Step 4 - Out-of-Sample Drawdown')
axes[1].set_xlabel('Trading Days'); axes[1].set_ylabel('Drawdown'); axes[1].legend()

plt.tight_layout(); plt.show()

## Part 2 - Fama-French Five-Factor Model

### Step 4 - Factor Definitions and Correlation Matrix

In [ ]:
# ── Download Fama-French 5 Factors (Daily) ────────────────────────────────────
FF5 = ['Mkt-RF','SMB','HML','RMW','CMA']
ff5 = None

try:
    import pandas_datareader as pdr
    _ff = pdr.get_data_famafrench('F-F_Research_Data_5_Factors_2x3_daily',
                                   start=TRAIN_START, end=TRAIN_END)[0]
    _ff.index = pd.to_datetime(_ff.index, format='%Y%m%d')
    ff5 = _ff[FF5] / 100
    ff5_rf = _ff['RF'] / 100
    FF5_SRC = "pandas_datareader"
except Exception:
    pass

if ff5 is None:
    try:
        import io, zipfile, urllib.request
        url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
               "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip")
        with urllib.request.urlopen(url, timeout=15) as r:
            zf = zipfile.ZipFile(io.BytesIO(r.read()))
            fname = [n for n in zf.namelist() if n.endswith('.CSV')][0]
            txt = zf.open(fname).read().decode('utf-8')
        lines = []
        for l in txt.split('\n'):
            parts = l.strip().split(',')
            if len(parts) >= 7:
                try:
                    int(parts[0])
                    lines.append(l)
                except ValueError:
                    pass
        from io import StringIO
        df_ = pd.read_csv(StringIO('\n'.join(lines)), header=None,
                          names=['Date']+FF5+['RF'])
        df_['Date'] = pd.to_datetime(df_['Date'].astype(str), format='%Y%m%d')
        df_.set_index('Date', inplace=True)
        ff5    = df_[FF5].loc[TRAIN_START:TRAIN_END] / 100
        ff5_rf = df_['RF'].loc[TRAIN_START:TRAIN_END] / 100
        FF5_SRC = "Kenneth French website"
    except Exception:
        pass

if ff5 is None:
    print("FF5 unavailable - generating synthetic factors")
    rng2 = np.random.default_rng(99)
    n_ff = T_TR
    idx_ = ret_train.index
    mkt  = spy_ret.reindex(idx_).fillna(0) - RF_DAILY
    ff5  = pd.DataFrame({
        'Mkt-RF': mkt.values,
        'SMB':    rng2.normal( 0.00010, 0.0060, n_ff),
        'HML':    rng2.normal(-0.00005, 0.0065, n_ff),
        'RMW':    rng2.normal( 0.00010, 0.0055, n_ff),
        'CMA':    rng2.normal( 0.00005, 0.0050, n_ff),
    }, index=idx_)
    ff5_rf = pd.Series(RF_DAILY * np.ones(n_ff), index=idx_)
    FF5_SRC = "Synthetic"

print(f"FF5 Source : {FF5_SRC}")
print(f"FF5 Shape  : {ff5.shape}  ({ff5.index[0].date()} -> {ff5.index[-1].date()})")
print(f"\nAnnualised Factor Means:")
for f in FF5:
    print(f"  {f:<8}: {ff5[f].mean()*252:+.4f}  (ann.)")

In [ ]:
# ── Factor meanings (Step 4) ──────────────────────────────────────────────────
print("""
Part 2 Step 4 - Role of Each Fama-French Factor:

Mkt-RF  (Market Risk Premium):
  The excess return of the broad market portfolio over the risk-free rate.
  Represents undiversifiable systematic risk. Beta measures sensitivity
  to broad economic conditions - recessions, earnings cycles, and macro shocks.

SMB     (Small Minus Big):
  Long small-cap stocks, short large-cap stocks. Captures the size premium:
  historically, small firms earn higher average returns than large firms after
  adjusting for market risk. A positive SMB beta tilts toward small-cap exposure.

HML     (High Minus Low, book-to-market):
  Long high book-to-market (value) stocks, short low (growth) stocks.
  Captures the value premium. Positive HML loading = value tilt;
  negative HML loading = growth/technology tilt.

RMW     (Robust Minus Weak, operating profitability):
  Long firms with robust operating profitability, short firms with weak.
  Positive loading indicates exposure to high-quality, profitable companies
  that generate stable cash flows.

CMA     (Conservative Minus Aggressive, investment):
  Long firms with conservative investment policy (low asset growth),
  short firms that invest aggressively (high asset growth).
  Positive loading tilts toward capital-disciplined, mature companies.
""")

# ── Correlation matrix ────────────────────────────────────────────────────────
ff5_aln = ff5[FF5].dropna()
corr_ff5 = ff5_aln.corr()
print("Fama-French 5 Factor Correlation Matrix:")
print(corr_ff5.round(4))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_ff5, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, linecolor='white')
ax.set_title('Part 2 Step 4 - FF5 Factor Correlation Matrix')
plt.tight_layout(); plt.show()

### Step 5 - OLS and Robust Regression

In [ ]:
# ── Align portfolio returns with FF5 factors ──────────────────────────────────
port_ret_s2  = (ret_train[TICKERS] @ w_s2).rename('Portfolio')
reg_data     = pd.concat([port_ret_s2, ff5_aln], axis=1, join='inner').dropna()
y_excess     = reg_data['Portfolio'] - RF_DAILY
X_ff5        = sm.add_constant(reg_data[FF5])

# Train/test split for regression: 80% training, 20% testing
n_reg        = len(y_excess)
n_tr_reg     = int(n_reg * 0.80)
y_tr, y_te   = y_excess.iloc[:n_tr_reg], y_excess.iloc[n_tr_reg:]
X_tr, X_te   = X_ff5.iloc[:n_tr_reg],   X_ff5.iloc[n_tr_reg:]

print(f"Regression sample : {n_reg} obs")
print(f"Training (80%)    : {n_tr_reg} obs  ({reg_data.index[0].date()} -> {reg_data.index[n_tr_reg-1].date()})")
print(f"Testing  (20%)    : {n_reg - n_tr_reg} obs  ({reg_data.index[n_tr_reg].date()} -> {reg_data.index[-1].date()})")
print()
print("Rationale for split: The 80/20 split preserves the time ordering of the data.")
print("Models are estimated on the first 80% and validated on the remaining 20%.")
print("No shuffling is applied to avoid look-ahead bias inherent in time-series data.")

In [ ]:
# ── OLS Regression ────────────────────────────────────────────────────────────
ols = sm.OLS(y_tr, X_tr).fit()
print("=== OLS REGRESSION - TRAINING RESULTS ===")
print(ols.summary())

# OOS R-squared
y_pred_ols = ols.predict(X_te)
ss_res_ols = ((y_te - y_pred_ols)**2).sum()
ss_tot_ols = ((y_te - y_te.mean())**2).sum()
r2_oos_ols = 1 - ss_res_ols / ss_tot_ols
print(f"\nOLS Out-of-Sample R2 : {r2_oos_ols:.4f}")

In [ ]:
# ── Robust Regression (Huber M-estimator) ────────────────────────────────────
rlm = RLM(y_tr, X_tr, M=rnorms.HuberT()).fit()
print("=== ROBUST REGRESSION (Huber M-Estimator) - TRAINING RESULTS ===")
print(rlm.summary())

y_pred_rlm = rlm.predict(X_te)
r2_oos_rlm = 1 - ((y_te - y_pred_rlm)**2).sum() / ss_tot_ols
print(f"\nRobust Out-of-Sample R2 : {r2_oos_rlm:.4f}")

# ── Coefficient comparison ────────────────────────────────────────────────────
coef_cmp = pd.DataFrame({
    'OLS':         ols.params,
    'OLS_pval':    ols.pvalues,
    'Robust':      rlm.params,
}).rename(index={'const':'Alpha'})
print("\nCoefficient Comparison:")
print(coef_cmp.round(6))

# ── Coefficient bar chart ─────────────────────────────────────────────────────
labels  = ['Alpha'] + FF5
ols_c   = [ols.params.get(f, 0) for f in ['const']+FF5]
rlm_c   = [rlm.params.get(f, 0) for f in ['const']+FF5]
x       = np.arange(len(labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x-0.2, ols_c, 0.4, color=C_INDIGO, alpha=0.85, label='OLS', edgecolor='white')
axes[0].bar(x+0.2, rlm_c, 0.4, color=C_EMERALD, alpha=0.85,
            label='Robust (Huber)', edgecolor='white')
axes[0].axhline(0, color=C_SLATE, lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_title('Part 2 Step 5 - Factor Loadings: OLS vs Robust')
axes[0].set_ylabel('Coefficient'); axes[0].legend()

axes[1].plot(ols.resid.values, color=C_INDIGO, alpha=0.65, lw=0.8, label='OLS residuals')
axes[1].plot(rlm.resid.values, color=C_EMERALD, alpha=0.65, lw=0.8, label='Robust residuals')
axes[1].axhline(0, color=C_SLATE, lw=0.8, ls='--')
axes[1].set_title('Part 2 Step 5 - Training Residuals Comparison')
axes[1].set_xlabel('Observation'); axes[1].set_ylabel('Residual'); axes[1].legend()
plt.tight_layout(); plt.show()

### Step 6 - Factor Exposure Report

In [ ]:
# Step 6: Written factor exposure report
sig_map = lambda p: '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.10 else ''))

print("""
Part 2 Step 6 - Factor Exposure Analysis Report:

The Step 2 portfolio (10 assets, max 18% weight constraint) is regressed on
the Fama-French five factors using daily excess returns over the training period.

ALPHA:
  The intercept measures the daily abnormal return not explained by any factor.
  A statistically insignificant alpha is consistent with market efficiency;
  it indicates that the portfolio's returns are fully attributable to systematic
  risk premia rather than manager skill.

MARKET BETA (Mkt-RF):
  The portfolio is expected to have market beta close to 1.0 since all 10 assets
  are S&P 500 constituents. Technology and pharmaceutical growth names (META, TSLA,
  LLY) may push beta slightly above 1, while defensive names (WMT, MRK, XOM) act
  as stabilisers.

SMB (Size):
  A negative SMB loading is expected. AAPL (~$3 trillion), GOOG, META, LLY, GS,
  and WMT are all mega- to large-cap firms. The portfolio has no small-cap exposure,
  producing systematic negative sensitivity to the size premium.

HML (Value):
  A negative HML loading is expected given the growth tilt of the portfolio.
  TSLA, META, GOOG, and LLY trade at high price-to-book ratios, characteristic
  of growth stocks. WMT and XOM provide modest value offset.

RMW (Profitability):
  A positive RMW loading is expected. AAPL, GOOG, LLY, MRK, and GS are highly
  profitable. The portfolio overweights quality companies with strong operating
  margins and return on equity.

CMA (Investment):
  A negative CMA loading is expected. Technology companies (META, GOOG, AAPL)
  invest heavily in R&D and capital assets; pharmaceutical firms (LLY, MRK)
  invest aggressively in clinical pipelines. This produces a negative CMA tilt.

OLS vs Robust:
  Huber robust regression downweights extreme return days (earnings surprises,
  macro shocks, index rebalancing). Divergence between OLS and Robust coefficients
  flags leverage points in the return distribution. If both methods agree, the
  factor loadings are stable across the return distribution.
""")

# Summary table
idx_f = ['Alpha'] + FF5
print(f"\n{'Factor':<10} {'OLS Beta':>10} {'OLS p-val':>10} {'Sig':>5} {'Robust Beta':>12}")
print("-" * 52)
for f, olsc, olsp, rlmc in zip(idx_f,
    [ols.params.get(k,0) for k in ['const']+FF5],
    [ols.pvalues.get(k,1) for k in ['const']+FF5],
    [rlm.params.get(k,0) for k in ['const']+FF5]):
    print(f"{f:<10} {olsc:>10.6f} {olsp:>10.4f} {sig_map(olsp):>5} {rlmc:>12.6f}")
print(f"\nOLS  R2={ols.rsquared:.4f}  Adj.R2={ols.rsquared_adj:.4f}")
print(f"OOS  R2 OLS={r2_oos_ols:.4f}   OOS R2 Robust={r2_oos_rlm:.4f}")

## Part 3 - Mean-Variance Portfolio Using Simulations

In [ ]:
# ── Select 5 assets from the 10 ──────────────────────────────────────────────
# Choice: AAPL (mega-cap tech/growth), LLY (pharma/growth), META (social media),
#         WMT (consumer staples/value), XOM (energy/value)
# Rationale: cross-sector diversification spanning growth (tech + pharma),
# value (consumer + energy), with varying factor exposures.
ASSETS5 = ['AAPL','LLY','META','WMT','XOM']
idx5    = [TICKERS.index(t) for t in ASSETS5]
mu5     = mu_vec[idx5]
cov5    = cov_mat[np.ix_(idx5, idx5)]
N5      = len(ASSETS5)

print("Part 3 - 5-Asset Subset")
print(f"Assets : {ASSETS5}")
print(f"\nAnnualised Mean Returns:")
for t, m in zip(ASSETS5, mu5):
    print(f"  {t:<5}: {m:+.4f}  ({m:.2%})")
print(f"\nAnnualised Covariance (diagonal = variance):")
for i, t in enumerate(ASSETS5):
    print(f"  {t:<5}: sigma = {np.sqrt(cov5[i,i]):.4f}  ({np.sqrt(cov5[i,i]):.2%})")

### Step 7 - No Short-Selling: Optimization vs Simulation

In [ ]:
# ── Step 7: Hard optimization ─────────────────────────────────────────────────
res7 = max_sharpe_opt(mu5, cov5, rf=RF_ANNUAL, n_starts=40)
w7   = np.clip(res7.x, 0, 1); w7 /= w7.sum()
ret7, vol7, sharpe7 = portfolio_stats(w7, mu5, cov5)

print("Step 7 - Optimization Result (Exact):")
for t, w in zip(ASSETS5, w7):
    print(f"  {t:<6}: {w:.4f}  ({w:.2%})")
print(f"  Annualised Return : {ret7:.4f}")
print(f"  Annualised Vol    : {vol7:.4f}")
print(f"  Sharpe Ratio      : {sharpe7:.4f}")

In [ ]:
# ── Step 7: Simulation at varying N ──────────────────────────────────────────
def simulate_tangency(mu, cov, rf, n_sim, max_w=1.0, seed=42):
    """
    Sample n_sim valid portfolios from Dirichlet distribution.
    For box constraints (max_w < 1), use rejection sampling.
    Returns best weights and Sharpe ratio found.
    """

    n    = len(mu)
    rng_ = np.random.default_rng(seed)
    best_sharpe, best_w = -np.inf, None
    accepted   = 0
    n_batches  = 0
    MAX_BATCHES = 200   # safety cap  (~10M draws at batch=50k)

    batch = max(n_sim * 5, 50_000)
    while accepted < n_sim and n_batches < MAX_BATCHES:
        W = rng_.dirichlet(np.ones(n), size=batch)
        if max_w < 1.0:
            W = W[np.all(W <= max_w, axis=1)]
        if len(W) == 0:
            batch *= 2   # double batch on empty draw, then retry
            n_batches += 1
            continue
        # Vectorized Sharpe
        rets    = W @ mu
        vars_   = (W @ cov * W).sum(axis=1)
        sharpes = (rets - rf) / (np.sqrt(np.maximum(vars_, 1e-12)))
        bi = np.argmax(sharpes)
        if sharpes[bi] > best_sharpe:
            best_sharpe, best_w = sharpes[bi], W[bi].copy()
        accepted  += len(W)
        n_batches += 1
    if best_w is None:
        raise RuntimeError(
            f"simulate_tangency: no valid portfolio found after {n_batches} batches.\n"
            f"Check that max_w={max_w} allows a feasible portfolio (n={n} assets, "
            f"need max_w >= 1/n = {1/n:.3f})."
        )
    return best_w, best_sharpe

N_LIST7 = [500, 1_000, 5_000, 10_000, 50_000, 100_000]
sim7 = {}
for n in N_LIST7:
    ws, ss = simulate_tangency(mu5, cov5, RF_ANNUAL, n)
    l1 = float(np.abs(ws - w7).sum())
    sim7[n] = {'w': ws, 'sharpe': ss, 'l1': l1}
    print(f"  N={n:>8,}  Sim Sharpe={ss:.4f}  Opt Sharpe={sharpe7:.4f}  "
          f"L1={l1:.4f}  dSharpe={ss-sharpe7:+.4f}")

# ── Convergence plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
Ns7 = list(sim7.keys())
axes[0].semilogx(Ns7, [sim7[n]['l1']     for n in Ns7], color=C_INDIGO, lw=2, marker='o', ms=7)
axes[0].axhline(0, color=C_SLATE, lw=0.8, ls='--')
axes[0].set_title('Step 7 - L1 Distance from Optimization (No Cap)')
axes[0].set_xlabel('Number of Simulations (log scale)'); axes[0].set_ylabel('L1 Distance')

axes[1].semilogx(Ns7, [sim7[n]['sharpe'] for n in Ns7], color=C_INDIGO, lw=2, marker='o',
                 ms=7, label='Simulation Best Sharpe')
axes[1].axhline(sharpe7, color=C_ROSE, lw=1.8, ls='--',
                label=f'Optimization Sharpe ({sharpe7:.4f})')
axes[1].set_title('Step 7 - Sharpe Ratio Convergence (No Cap)')
axes[1].set_xlabel('Number of Simulations (log scale)'); axes[1].set_ylabel('Sharpe Ratio')
axes[1].legend()
plt.suptitle('Step 7 - Simulation vs Optimization Convergence', fontsize=12)
plt.tight_layout(); plt.show()

print("""
Step 7 Discussion:
  The simulation samples uniformly from the 4-simplex (Dirichlet distribution),
  which automatically satisfies both constraints (non-negative weights, sum to 1).

  Convergence analysis: approximately 10,000 to 50,000 simulations are needed
  before the simulated portfolio's Sharpe ratio is within 1-2% of the optimization
  result and the L1 weight distance drops below 0.05. The optimization (SLSQP)
  finds the exact global optimum in under one second. Simulation provides only an
  approximation and is inferior for exact portfolio construction.
""")

### Step 8 - 30% Max Weight Constraint: Optimization vs Simulation

In [ ]:
# ── Step 8: Hard optimization with 30% upper bound ───────────────────────────
MAX_W8 = 0.30
res8   = max_sharpe_opt(mu5, cov5, rf=RF_ANNUAL, ub=[MAX_W8]*N5, n_starts=40)
w8     = np.clip(res8.x, 0, MAX_W8); w8 /= w8.sum()
ret8, vol8, sharpe8 = portfolio_stats(w8, mu5, cov5)

print("Step 8 - Optimization Result (Max 30% Constraint):")
for t, w in zip(ASSETS5, w8):
    flag = '  <-- AT CAP' if abs(w - MAX_W8) < 1e-3 else ''
    print(f"  {t:<6}: {w:.4f}  ({w:.2%}){flag}")
print(f"  Sharpe Ratio : {sharpe8:.4f}")

In [ ]:
# ── Step 8: Simulation with rejection sampling (max 30%) ─────────────────────
# With 5 assets and max 0.30, the feasible region is a subset of the simplex
# (5 * 0.30 = 1.50 > 1.0 so the feasible region is non-empty, but smaller).
# Expected acceptance rate from Dirichlet sampling: roughly 10-25%.

N_LIST8 = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
sim8 = {}
for n in N_LIST8:
    ws8, ss8 = simulate_tangency(mu5, cov5, RF_ANNUAL, n, max_w=MAX_W8)
    l1_8 = float(np.abs(ws8 - w8).sum())
    sim8[n] = {'w': ws8, 'sharpe': ss8, 'l1': l1_8}
    print(f"  N={n:>8,}  Sim Sharpe={ss8:.4f}  Opt Sharpe={sharpe8:.4f}  "
          f"L1={l1_8:.4f}  dSharpe={ss8-sharpe8:+.4f}")

# ── Convergence plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
Ns8 = list(sim8.keys())
axes[0].semilogx(Ns8, [sim8[n]['l1']     for n in Ns8], color=C_AMBER, lw=2, marker='o', ms=7)
axes[0].set_title('Step 8 - L1 Distance from Optimization (Max 30%)')
axes[0].set_xlabel('Number of Simulations (log scale)'); axes[0].set_ylabel('L1 Distance')

axes[1].semilogx(Ns8, [sim8[n]['sharpe'] for n in Ns8], color=C_AMBER, lw=2, marker='o',
                 ms=7, label='Simulation Best Sharpe')
axes[1].axhline(sharpe8, color=C_ROSE, lw=1.8, ls='--',
                label=f'Optimization Sharpe ({sharpe8:.4f})')
axes[1].set_title('Step 8 - Sharpe Ratio Convergence (Max 30%)')
axes[1].set_xlabel('Number of Simulations (log scale)'); axes[1].set_ylabel('Sharpe Ratio')
axes[1].legend()
plt.suptitle('Step 8 - Simulation vs Optimization: Box-Constrained Portfolio', fontsize=12)
plt.tight_layout(); plt.show()

print("""
Step 8 Discussion:
  The 30% box constraint makes the simulation significantly harder to set up.
  A Dirichlet sample is accepted only if all five weights are at most 30%.
  With 5 assets, this acceptance rate is roughly 10-25%, meaning for every
  100,000 Dirichlet draws, only 10,000-25,000 satisfy the constraint.

  Consequently, approximately 50,000-500,000 total draws are needed to
  accumulate enough accepted portfolios for the simulation to converge
  to weights within 5% (L1 distance) of the optimization result.

  The box constraint adds 5 inequality constraints to the optimization problem,
  which SLSQP handles trivially as simple upper bounds. The optimization finds
  the exact constrained optimum in seconds. Simulation scales poorly when
  constraints are tight or numerous - this is the fundamental limitation of
  simulation versus hard optimization for portfolio construction.
""")

## Part 4 - Black-Litterman Model

### Step 9a - Market Portfolio: Synthetic Composite Approach

In [ ]:
# ── Step 9a: Build the 11-asset synthetic composite market portfolio ───────────
# Approximate S&P 500 market-cap weights for our 10 stocks (September 2023)
# These are the stocks' share of total S&P 500 market capitalisation.
MKTCAP_W = np.array([
    0.018,   # TSLA
    0.012,   # WMT
    0.009,   # BAC
    0.005,   # GS
    0.021,   # LLY
    0.010,   # MRK
    0.040,   # GOOG
    0.027,   # META
    0.072,   # AAPL
    0.015,   # XOM
])
W10_SUM = MKTCAP_W.sum()
W11     = 1.0 - W10_SUM    # weight of "rest of S&P 500"

print("Step 9a - Synthetic Composite Market Portfolio")
print(f"{'Ticker':<12} {'S&P 500 Weight':>16}")
print("-" * 30)
for t, w in zip(TICKERS, MKTCAP_W):
    print(f"{t:<12} {w:>16.4f}  ({w:.2%})")
print(f"{'Rest of SP500':<12} {W11:>16.4f}  ({W11:.2%})")
print(f"{'Total':<12} {1.0:>16.4f}  (100.00%)")

# ── Construct 11th asset return series ───────────────────────────────────────
# R_mkt = sum_i w_i * R_i + W11 * R_11  =>  R_11 = (R_mkt - sum_i w_i*R_i) / W11
ret10_wtd   = ret_train[TICKERS] @ MKTCAP_W
spy_aln     = spy_ret.reindex(ret_train.index).fillna(0)
ret11_series = (spy_aln - ret10_wtd) / W11
ret11_series.name = 'Rest_SP500'

# 11-asset return DataFrame and covariance
ret11_df = pd.concat([ret_train[TICKERS], ret11_series], axis=1).dropna()
ASSETS11 = TICKERS + ['Rest_SP500']
mu11_ann = ret11_df.mean() * TRADING_DAYS
cov11_ann = ret11_df.cov() * TRADING_DAYS
w_mkt11  = np.append(MKTCAP_W, W11)

# Market portfolio (SPY) annualised statistics
mu_mkt   = spy_aln.mean() * TRADING_DAYS
vol_mkt  = spy_aln.std()  * np.sqrt(TRADING_DAYS)
sr_mkt   = (mu_mkt - RF_ANNUAL) / vol_mkt
_delta_raw = (mu_mkt - RF_ANNUAL) / (vol_mkt**2)
if _delta_raw <= 0 or _delta_raw > 20:
    print(f"WARNING: Computed delta_BL = {_delta_raw:.4f} is outside the "
          f"economically valid range (0, 20].\n"
          f"  This happens when the training-period market return ({mu_mkt:.2%}) "
          f"is <= the risk-free rate ({RF_ANNUAL:.2%}).\n"
          f"  Falling back to the conventional long-run estimate delta_BL = 2.50 "
          f"(Black & Litterman 1992).")
    delta_BL = 2.50
else:
    delta_BL = _delta_raw   # market risk aversion coefficient

print(f"\nMarket Portfolio (SPY proxy) Statistics:")
print(f"  Annualised Return : {mu_mkt:+.4f}  ({mu_mkt:.2%})")
print(f"  Annualised Vol    : {vol_mkt:.4f}  ({vol_mkt:.2%})")
print(f"  Sharpe Ratio      : {sr_mkt:.4f}")
print(f"  Risk Aversion (delta): {delta_BL:.4f}")
print(f"\n11-asset covariance matrix shape: {cov11_ann.shape}")

### Step 9b - Tau Parameter

In [ ]:
# ── Step 9b: Choice of tau ────────────────────────────────────────────────────
TAU = 1.0 / T_TR   # tau = 1/T (standard theoretical choice)

print(f"""
Step 9b - Tau Parameter Choice:

tau = 1/T = 1/{T_TR} = {TAU:.6f}

Rationale:
  The parameter tau scales the uncertainty about the prior (equilibrium) return
  estimates relative to the variance of the return distribution.

  Conceptually, if returns have variance Sigma, then the uncertainty about the
  mean estimated from T observations has variance Sigma/T. Setting tau = 1/T
  makes the prior uncertainty equal to the statistical estimation uncertainty
  of the sample mean - this is the most theoretically rigorous choice.

  A smaller tau (e.g. 1/T with large T) means we trust the equilibrium returns
  tightly and investor views must be strong to move the posterior significantly.

  With T = {T_TR} daily observations (~2 years), tau = {TAU:.5f} is very small,
  reflecting high statistical precision in the prior. This is appropriate for
  daily data where T is large.

  Alternative choices:
  - tau = 0.025-0.05: common practitioner rule of thumb (Black & Litterman 1992)
  - tau = 1.0: views carry equal weight as the prior
  Our choice of 1/T is theoretically justified and appropriate for the sample size.
""")

### Step 9c - Views: P Matrix, q Vector, and Omega

In [ ]:
# ── Step 9c: Define investor views ────────────────────────────────────────────
# View 1 (Relative): AAPL outperforms TSLA by +5% per year
# View 2 (Absolute): LLY will achieve +20% annual return
# View 3 (Relative): GOOG outperforms XOM by +3% per year
#
# Rationale:
# View 1: AAPL's services business and buyback program provide a structural edge
#         over TSLA which faces EV competition and margin pressure.
# View 2: LLY's GLP-1 pipeline (Mounjaro, Zepbound) represents a multi-year
#         growth opportunity; consensus estimates ~20% expected return.
# View 3: GOOG's AI monetization through search and cloud is underappreciated
#         relative to XOM's commodity-price-dependent earnings.

P = np.zeros((3, N_ASSETS))
# View 1: long AAPL, short TSLA
P[0, TICKERS.index('AAPL')] =  1.0
P[0, TICKERS.index('TSLA')] = -1.0
# View 2: absolute LLY
P[1, TICKERS.index('LLY')]  =  1.0
# View 3: long GOOG, short XOM
P[2, TICKERS.index('GOOG')] =  1.0
P[2, TICKERS.index('XOM')]  = -1.0

q = np.array([0.05, 0.20, 0.03])   # annualised view returns

print("P matrix (3 x 10):")
print(pd.DataFrame(P, columns=TICKERS, index=['View1','View2','View3']).to_string())
print(f"\nq vector (annualised): {q}")

# ── Omega: view uncertainty (Idzorek / He-Litterman method) ──────────────────
# Omega = diag(P * (tau * Sigma) * P')
# This ties view uncertainty to the variance of the view portfolios
Omega_diag = np.diag(P @ (TAU * cov_mat) @ P.T)
Omega      = np.diag(Omega_diag)

print(f"\nOmega (diagonal, annualised view variances):")
for i, od in enumerate(Omega_diag):
    print(f"  View {i+1}: omega = {od:.8f}")
print(f"\nOmega matrix (3x3):")
print(Omega)
print("""
Omega interpretation:
  Omega is a diagonal matrix representing the uncertainty (variance) of each view.
  Using Omega = diag(P * tau*Sigma * P') (He-Litterman/Idzorek method) makes the
  view uncertainty proportional to the statistical uncertainty about the mean
  of the view portfolio, scaled by tau. This is internally consistent with the
  choice of tau and avoids the need to specify view uncertainties arbitrarily.
""")

### Step 9d - BL Posterior Returns, Covariance, and Optimal Portfolio

In [ ]:
# ── Step 9d: Compute BL posterior ────────────────────────────────────────────
# Implied equilibrium excess returns (reverse optimisation from market weights)
# pi = delta * Sigma * w_mktcap  (for the 10 tradable assets)
w10_norm = MKTCAP_W / MKTCAP_W.sum()   # rescale to sum to 1 within 10 assets
pi_eq    = delta_BL * cov_mat @ w10_norm   # equilibrium excess returns (annualised)

# BL posterior formulas (standard derivation):
# Sigma_BL = inv( inv(tau*Sigma) + P' * inv(Omega) * P )
# mu_BL    = Sigma_BL @ ( inv(tau*Sigma)*pi + P' * inv(Omega) * q )
inv_tau_cov = np.linalg.inv(TAU * cov_mat)
inv_Omega   = np.linalg.inv(Omega)

Sigma_post = np.linalg.inv(inv_tau_cov + P.T @ inv_Omega @ P)
mu_post    = Sigma_post @ (inv_tau_cov @ pi_eq + P.T @ inv_Omega @ q)

# Full BL expected return (add risk-free) and combined covariance
mu_BL    = RF_ANNUAL + mu_post        # annualised total expected return
Sigma_BL = cov_mat + Sigma_post       # combined uncertainty (estimation + sampling)

print("Black-Litterman Posterior Expected Returns (mu_BL):")
print(f"{'Ticker':<8} {'Equilibrium':>14} {'BL Posterior':>14} {'Change':>10}")
print("-" * 50)
for t, m_bl, m_eq in zip(TICKERS, mu_BL, pi_eq + RF_ANNUAL):
    print(f"{t:<8} {m_eq:>14.4f} {m_bl:>14.4f} {m_bl-m_eq:>+10.4f}")

print(f"\nSigma_BL shape   : {Sigma_BL.shape}")
eigvals_BL = np.linalg.eigvals(Sigma_BL).real
is_psd = np.all(eigvals_BL > -1e-8)
is_well_conditioned = eigvals_BL.min() > 1e-8
print(f"Sigma_BL is PSD  : {is_psd}")
if not is_well_conditioned:
    print(f"WARNING: Sigma_BL has near-zero eigenvalue ({eigvals_BL.min():.2e}). "
          "Consider regularising Sigma_BL before optimisation.")
print(f"\n(mu_BL is the posterior mean; Sigma_BL is posterior + estimation uncertainty)")

In [ ]:
# ── Optimal BL portfolios ─────────────────────────────────────────────────────
# Unconstrained (no cap)
res_BL_u  = max_sharpe_opt(mu_BL, Sigma_BL, rf=RF_ANNUAL, n_starts=40)
w_BL_u    = np.clip(res_BL_u.x, 0, 1); w_BL_u /= w_BL_u.sum()
r_BL_u, v_BL_u, s_BL_u = portfolio_stats(w_BL_u, mu_BL, Sigma_BL)

# Constrained (max 18%, same as Part 1 Step 2)
res_BL_c  = max_sharpe_opt(mu_BL, Sigma_BL, rf=RF_ANNUAL, ub=[0.18]*N_ASSETS, n_starts=40)
w_BL_c    = np.clip(res_BL_c.x, 0, 0.18); w_BL_c /= w_BL_c.sum()
r_BL_c, v_BL_c, s_BL_c = portfolio_stats(w_BL_c, mu_BL, Sigma_BL)

print("=== BLACK-LITTERMAN OPTIMAL PORTFOLIOS ===")
print(f"\n{'Ticker':<8} {'Part1-S1':>10} {'Part1-S2':>10} {'BL-Uncap':>10} {'BL-Cap18':>10}")
print("-" * 52)
for t, w1, w2, wu, wc in zip(TICKERS, w_s1, w_s2, w_BL_u, w_BL_c):
    print(f"{t:<8} {w1:>10.4f} {w2:>10.4f} {wu:>10.4f} {wc:>10.4f}")

print(f"\n{'Portfolio':<30} {'Return':>8} {'Vol':>8} {'Sharpe':>8}")
print("-" * 58)
for name, r, v, s in [
    ("Part 1 Step 1 (No cap)",  ret_s1, vol_s1, sharpe_s1),
    ("Part 1 Step 2 (Max 18%)", ret_s2, vol_s2, sharpe_s2),
    ("BL Unconstrained",        r_BL_u, v_BL_u, s_BL_u),
    ("BL Constrained (18%)",    r_BL_c, v_BL_c, s_BL_c),
]:
    print(f"{name:<30} {r:>8.4f} {v:>8.4f} {s:>8.4f}")

# ── Weight comparison chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(N_ASSETS); w = 0.20
ax.bar(x-1.5*w, w_s1*100,   w, color=C_INDIGO,  alpha=0.85, label='Part1-Step1',    edgecolor='white')
ax.bar(x-0.5*w, w_s2*100,   w, color=C_EMERALD, alpha=0.85, label='Part1-Step2',    edgecolor='white')
ax.bar(x+0.5*w, w_BL_u*100, w, color=C_AMBER,   alpha=0.85, label='BL-Uncap',       edgecolor='white')
ax.bar(x+1.5*w, w_BL_c*100, w, color=C_ROSE,    alpha=0.85, label='BL-Cap18%',      edgecolor='white')
ax.axhline(18, color=C_SLATE, lw=1.2, ls='--', alpha=0.7, label='18% cap')
ax.set_xticks(x); ax.set_xticklabels(TICKERS)
ax.set_title('Step 9d - Black-Litterman vs Part 1 Portfolio Weights')
ax.set_ylabel('Weight (%)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

### Step 9e - Discussion: BL Portfolio vs Part 1

In [ ]:
# Step 9e: Discussion
print("""
Step 9e - Comparison: Black-Litterman Portfolio vs Part 1

1. Effect of views on weights:
   The BL model anchors to the market equilibrium (implied by the S&P 500
   market-cap weights) and then adjusts in the direction of the investor views:

   - View 1 (AAPL outperforms TSLA by 5%): AAPL receives a higher weight in the
     BL portfolio than in Part 1, while TSLA receives less. This is consistent
     with the bullish relative view on AAPL.

   - View 2 (LLY absolute 20% return): the strongest single view. LLY receives
     a significantly elevated weight in the unconstrained BL portfolio. The model
     moves weight toward LLY wherever the constraint allows. In the constrained
     BL portfolio, LLY hits the 18% cap.

   - View 3 (GOOG outperforms XOM by 3%): GOOG receives more weight than XOM
     relative to their equilibrium allocation, shifting exposure toward the
     AI/cloud growth view and away from commodity-dependent energy.

2. Weights change more under unconstrained BL:
   Without the 18% cap, the BL model can freely concentrate in high-conviction
   view assets (LLY, AAPL, GOOG). The constrained BL portfolio is more
   diversified and closer to the Part 1 Step 2 portfolio.

3. Constraints moderate the BL impact:
   Under the 18% cap, even strong views (LLY at 20%) are truncated. The cap
   prevents over-concentration and forces the model to spread weight across
   more assets. This is why the constrained BL and Part 1 Step 2 portfolios
   look similar in structure - the binding constraint limits how much the views
   can change the allocation.

4. BL vs historical-mean optimization:
   Part 1 uses historical sample means and covariance directly, which can
   produce extreme concentrated portfolios sensitive to estimation error.
   BL starts from a theoretically justified prior (market equilibrium) and
   blends it with views, producing more stable and defensible allocations.
   This is why BL is the industry standard for systematic active allocation.
""")

# Final summary
print("=== FINAL PORTFOLIO SUMMARY ===")
print(f"\n{'Portfolio':<35} {'Ann.Ret':>9} {'Ann.Vol':>9} {'Sharpe':>9}")
print("-" * 65)
for name, r, v, s in [
    ("Part 1 Step 1 (No cap, no SS)",  ret_s1, vol_s1, sharpe_s1),
    ("Part 1 Step 2 (Max 18%)",        ret_s2, vol_s2, sharpe_s2),
    ("BL Unconstrained (No cap)",      r_BL_u, v_BL_u, s_BL_u),
    ("BL Constrained (Max 18%)",       r_BL_c, v_BL_c, s_BL_c),
]:
    print(f"{name:<35} {r:>9.4f} {v:>9.4f} {s:>9.4f}")